# How did the COVID-19 pandemic impact the economy? Exploratory data analysis

**Countries studied (7):** Brazil, China, France, Germany, India, Rwanda, United States
**Economy:** 2018-2024 · **Health and policy:** 2020-2022

The dashboard's central research question concerns the global economy. The data available covers **seven countries only**, so every finding below describes these seven countries and cannot be generalized to the world.

| Theme | Dashboard question | Main data |
|---|---|---|
| 1. Economic growth | How did GDP and economic output change? | GDP growth |
| 2. Employment | How did unemployment and jobs change? | Unemployment rate |
| 3. Prices and inflation | How did inflation and consumer prices evolve? | Inflation |
| 4. Global differences | How did the impact vary across countries? | All files, plus health and policy context |
| 5. Economic recovery | How did economies recover after the initial shock? | GDP, unemployment, inflation 2022-2024 |

The notebook contains summary-statistics tables, charts with an explanation of what each reveals, evidence-based findings for the five themes, and a short findings document written automatically at the end.

**To run it:** upload the five CSV files next to the notebook (in Colab: folder icon in the left sidebar, then upload) and run the cells from top to bottom.

## 1. Setup and visual style

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import spearmanr
from IPython.display import display, Markdown

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)

# ---- One palette for every chart ----
INK, MUTED, GRID = "#2B2D42", "#6B7280", "#E5E7EB"
COUNTRY_COLORS = {"Brazil": "#009E73", "China": "#D55E00", "France": "#0072B2", "Germany": "#4D4D4D",
                  "India": "#E69F00", "Rwanda": "#CC79A7", "United States": "#56B4E9"}
YEAR_COLORS = {"2020": "#C0392B", "2021": "#2E86AB", "2022": "#8D99AE"}
IND_COLORS  = {"GDP growth (%)": "#0072B2", "Inflation (%)": "#D55E00", "Unemployment (%)": "#009E73"}
BAD, GOOD = "#D55E00", "#0072B2"
SHADE = "rgba(192, 57, 43, 0.10)"
COUNTRIES = list(COUNTRY_COLORS)
FIGS = {}

# ---- Small helpers ----
def mon(ts): return pd.Timestamp(ts).strftime("%B %Y")
fmt0 = lambda v: f"{v:,.0f}"
nm = lambda c: "the United States" if c == "United States" else c
Nm = lambda c: "The United States" if c == "United States" else c
def join_names(names):
    names = [nm(c) for c in names]
    return "none" if not names else names[0] if len(names) == 1 else ", ".join(names[:-1]) + " and " + names[-1]

def polish(fig, title, subtitle=None, height=520, legend=True, facets=False):
    t = f"<b>{title}</b>"
    if subtitle:
        t += f"<br><span style='font-size:13px;color:{MUTED}'>{subtitle}</span>"
    fig.update_layout(template="plotly_white", height=height,
        font=dict(family="Arial, sans-serif", size=13, color=INK),
        title=dict(text=t, x=0, xanchor="left"), margin=dict(t=115, l=60, r=30, b=60),
        paper_bgcolor="white", plot_bgcolor="white", showlegend=legend,
        legend=dict(orientation="h", y=1.02, yanchor="bottom", x=0, title_text=""), hoverlabel=dict(font_size=12))
    fig.update_xaxes(showgrid=False, linecolor="#9CA3AF")
    fig.update_yaxes(gridcolor=GRID, zerolinecolor="#9CA3AF")
    if facets:
        fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], font=dict(size=13, color=INK)))
    return fig

def style_table(df, gradients=None, formats=None):
    s = df.style.format(formats or "{:,.1f}")
    for cols, cmap in (gradients or []):
        s = s.background_gradient(subset=cols, cmap=cmap)
    return s.set_table_styles([{"selector": "th", "props": [("background-color", "#F3F4F6"), ("color", INK), ("font-weight", "600")]}])

def md_table(df, index_name="Country"):
    """Turn a DataFrame of ready-to-print values into a Markdown table."""
    cols = [index_name] + list(df.columns)
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for idx_, row in df.iterrows():
        out.append("| " + " | ".join([str(idx_)] + [str(v) for v in row.values]) + " |")
    return "\n".join(out)

## 2. Loading the data

The files use different separators (`,` or `;`), so the loader reads the first line of each file and picks the separator itself. Files are found by keyword, so small differences in file names do not matter.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
gdp   = pd.read_csv("gdp_growth_pct_annual.csv",    sep=",")
infl  = pd.read_csv("inflation_pct_annual.csv", sep=";")
unemp = pd.read_csv("unemployment_pct_annual.csv",  sep=";")
y = pd.read_csv("merged_dataset_by_year.csv",  sep=";")
m = pd.read_csv("merged_dataset_by_month.csv", sep=";")

# 2) Economy table, 2018-2024 (49 rows = 7 countries x 7 years)
eco = (gdp.merge(infl, on=["country", "code", "year"])
          .merge(unemp, on=["country", "code", "year"])
          .sort_values(["country", "year"]).reset_index(drop=True))

# 3) Small preparations
m["date"] = pd.to_datetime(m["month"])
y["year_str"] = y["year"].astype(str)

# 4) Master table for Act 3: economy + health/policy + pre-COVID baseline
health_cols = ["new_cases_per_million", "new_deaths_per_million",
               "sti_vaccinated", "sti_nonvaccinated", "sti_weighted"]
full = eco.merge(y[["country", "code", "year"] + health_cols],
                 on=["country", "code", "year"], how="left")   # health is empty outside 2020-2022
base = (eco[eco.year.isin([2018, 2019])]
        .groupby("country")[["gdp_growth_pct", "inflation_pct", "unemployment_pct"]]
        .mean().add_suffix("_base").reset_index())
full = full.merge(base, on="country")
full["gdp_effect"]   = full["gdp_growth_pct"]   - full["gdp_growth_pct_base"]    # points below/above own normal
full["unemp_effect"] = full["unemployment_pct"] - full["unemployment_pct_base"]
full["infl_effect"]  = full["inflation_pct"]    - full["inflation_pct_base"]

cov = full[full.year.between(2020, 2022)].copy()               # the 21 rows where all data exist
cov["year_str"] = cov["year"].astype(str)

print(eco.shape, y.shape, m.shape, cov.shape)   # (49, 6) (21, 12) (252, 12) (21, 18)

## 3. Data checks

In [ ]:
overview = pd.DataFrame({
    "File": ["GDP growth", "Inflation", "Unemployment", "Health + policy + economy (yearly)", "Health + policy + economy (monthly)"],
    "Rows": [len(gdp), len(infl), len(unemp), len(y), len(m)],
    "Countries": [d["country"].nunique() for d in (gdp, infl, unemp, y, m)],
    "First period": [d["year"].min() if "year" in d else d["month"].min() for d in (gdp, infl, unemp, y, m)],
    "Last period": [d["year"].max() if "year" in d else d["month"].max() for d in (gdp, infl, unemp, y, m)],
    "Missing cells": [int(d.isna().sum().sum()) for d in (gdp, infl, unemp, y, m)]})
overview[["First period", "Last period"]] = overview[["First period", "Last period"]].astype(str)
display(style_table(overview, formats={"Rows": "{:,.0f}", "Countries": "{:,.0f}", "Missing cells": "{:,.0f}"}).hide(axis="index"))

m["date"] = pd.to_datetime(m["month"])
m["year"] = m["date"].dt.year
chk = (m.groupby(["country", "year"]).agg(cases=("new_cases_per_million", "sum"), deaths=("new_deaths_per_million", "sum"),
                                          sti=("sti_weighted", "mean")).reset_index().merge(y, on=["country", "year"]))
print("Largest difference between the monthly and yearly files  ->  cases:", round((chk.cases - chk.new_cases_per_million).abs().max(), 3),
      "| deaths:", round((chk.deaths - chk.new_deaths_per_million).abs().max(), 3),
      "| stringency:", round((chk.sti - chk.sti_weighted).abs().max(), 3))
print("Countries:", sorted(y["country"].unique()))

The monthly file is a finer version of the yearly one (cases and deaths add up exactly, stringency averages out). Its economic columns are the yearly value repeated twelve times, so the **monthly file is used only for the shape of the epidemic and of policy over time**, and the **yearly file for every comparison with the economy**.

## 4. Preparing the analysis tables

- `eco`: economy, 2018-2024 (49 rows).
- `full` / `cov`: economy joined with health and policy for 2020-2022 (21 rows). Each country's own **2018-2019 average** is the pre-COVID baseline, and an *effect* is a value minus that baseline, which compares each country with its own normal.
- `idx`: size of each economy with 2019 = 100, and the size it would have reached had 2018-2019 growth continued (an estimate).

In [ ]:
y["year_str"] = y["year"].astype(str)
m["sti_gap"] = m["sti_nonvaccinated"] - m["sti_vaccinated"]

ECON = {"gdp_growth_pct": "GDP growth (%)", "inflation_pct": "Inflation (%)", "unemployment_pct": "Unemployment (%)"}
eco = (gdp.merge(infl, on=["country", "code", "year"]).merge(unemp, on=["country", "code", "year"])
          .sort_values(["country", "year"]).reset_index(drop=True))

def period_of(yr):
    if yr <= 2019: return "2018-19 (before COVID)"
    if yr <= 2021: return "2020-21 (shock)"
    if yr == 2022: return "2022"
    return "2023-24 (after)"
PERIODS = ["2018-19 (before COVID)", "2020-21 (shock)", "2022", "2023-24 (after)"]
eco["period"] = eco["year"].map(period_of)

health_cols = ["new_cases_per_million", "new_deaths_per_million", "sti_vaccinated", "sti_nonvaccinated", "sti_weighted"]
full = eco.merge(y[["country", "code", "year"] + health_cols], on=["country", "code", "year"], how="left")
base = eco[eco.year.isin([2018, 2019])].groupby("country")[list(ECON)].mean().add_suffix("_base").reset_index()
full = full.merge(base, on="country")
full["gdp_effect"]   = full["gdp_growth_pct"]   - full["gdp_growth_pct_base"]
full["unemp_effect"] = full["unemployment_pct"] - full["unemployment_pct_base"]
full["infl_effect"]  = full["inflation_pct"]    - full["inflation_pct_base"]
cov = full[full.year.between(2020, 2022)].copy()
cov["year_str"] = cov["year"].astype(str)

idx = eco[eco.year >= 2019].copy()
idx["factor"] = 1 + idx["gdp_growth_pct"] / 100
idx.loc[idx.year == 2019, "factor"] = 1.0
idx["gdp_index"] = 100 * idx.groupby("country")["factor"].cumprod()
trend_growth = eco[eco.year.isin([2018, 2019])].groupby("country")["gdp_growth_pct"].mean().rename("trend_growth")
idx = idx.merge(trend_growth, on="country")
idx["trend_index"] = 100 * (1 + idx["trend_growth"] / 100) ** (idx["year"] - 2019)
idx["gap_pct"] = idx["gdp_index"] / idx["trend_index"] * 100 - 100

# frequently used pivots
g = eco.pivot(index="country", columns="year", values="gdp_growth_pct")
u = eco.pivot(index="country", columns="year", values="unemployment_pct")
inf = eco.pivot(index="country", columns="year", values="inflation_pct")
print(eco.shape, y.shape, m.shape, cov.shape, idx.shape)

## 5. Summary-statistics tables (the seven countries)

**Table 1. The typical economy by period** (median across the seven countries).

In [ ]:
t1 = eco.groupby("period")[list(ECON)].median().reindex(PERIODS).rename(columns=ECON)
display(style_table(t1, formats="{:.1f}", gradients=[(["GDP growth (%)"], "RdBu"), (["Inflation (%)"], "Oranges"), (["Unemployment (%)"], "Greens")]))

**Table 2. Economy by country, 2018-2024.**

In [ ]:
rows = []
for c, gg in eco.groupby("country"):
    rows.append({"Country": c, "Average GDP growth (%)": gg.gdp_growth_pct.mean(), "Worst GDP growth (%)": gg.gdp_growth_pct.min(),
                 "Worst year": int(gg.loc[gg.gdp_growth_pct.idxmin(), "year"]), "Peak inflation (%)": gg.inflation_pct.max(),
                 "Peak inflation year": int(gg.loc[gg.inflation_pct.idxmax(), "year"]), "Peak unemployment (%)": gg.unemployment_pct.max(),
                 "Peak unemployment year": int(gg.loc[gg.unemployment_pct.idxmax(), "year"])})
t2 = pd.DataFrame(rows).set_index("Country")
fmt = {c: "{:.1f}" for c in t2.columns}; fmt.update({c: "{:d}" for c in t2.columns if "year" in c})
display(style_table(t2, formats=fmt, gradients=[(["Worst GDP growth (%)"], "RdBu"), (["Peak inflation (%)"], "Oranges"), (["Peak unemployment (%)"], "Greens")]))

**Table 3. Health and policy by country, 2020-2022.**

In [ ]:
pk_d = m.loc[m.groupby("country")["new_deaths_per_million"].idxmax()].set_index("country")
pk_s = m.loc[m.groupby("country")["sti_weighted"].idxmax()].set_index("country")
mx_g = m.loc[m.groupby("country")["sti_gap"].idxmax()].set_index("country")
t3 = pd.DataFrame({
    "Reported deaths per million, 2020-22": y.groupby("country")["new_deaths_per_million"].sum(),
    "Reported cases per million, 2020-22":  y.groupby("country")["new_cases_per_million"].sum(),
    "Deadliest month": pk_d["date"].dt.strftime("%b %Y"),
    "Average stringency (0-100)": y.groupby("country")["sti_weighted"].mean(),
    "Strictest month": pk_s["date"].dt.strftime("%b %Y"),
    "Peak stringency": pk_s["sti_weighted"],
    "Largest gap, non-vaccinated vs vaccinated (pts)": mx_g["sti_gap"]})
fmt = {"Reported deaths per million, 2020-22": "{:,.0f}", "Reported cases per million, 2020-22": "{:,.0f}",
       "Average stringency (0-100)": "{:.1f}", "Peak stringency": "{:.1f}", "Largest gap, non-vaccinated vs vaccinated (pts)": "{:.1f}"}
display(style_table(t3, formats=fmt, gradients=[(["Reported deaths per million, 2020-22"], "Reds"), (["Average stringency (0-100)"], "YlOrRd")]))

**Table 4. Economic impact by country** (each measured against the country's own 2018-2019 level or trend).

In [ ]:
impact = pd.DataFrame({
    "GDP growth vs own 2018-19 average, 2020 (pts)": full[full.year == 2020].set_index("country")["gdp_effect"],
    "Unemployment change, 2019 to 2020 (pts)": u[2020] - u[2019],
    "Inflation vs own 2018-19 average, 2022 (pts)": full[full.year == 2022].set_index("country")["infl_effect"],
    "Economy vs 2018-19 trend, 2024 (%)": idx[idx.year == 2024].set_index("country")["gap_pct"]})
display(style_table(impact, formats="{:+.1f}", gradients=[
    (["GDP growth vs own 2018-19 average, 2020 (pts)", "Economy vs 2018-19 trend, 2024 (%)"], "RdBu"),
    (["Unemployment change, 2019 to 2020 (pts)", "Inflation vs own 2018-19 average, 2022 (pts)"], "RdBu_r")]))

---
# Overview: the typical economy
*A first look at the three economic indicators together, before the five themes.*

### Chart 1. The typical economy: output and jobs fell in 2020, prices rose in 2022

In [ ]:
med = eco.groupby("year")[list(ECON)].median().reset_index().melt(id_vars="year", var_name="indicator", value_name="median")
med["indicator"] = med["indicator"].map(ECON)
fig = px.line(med, x="year", y="median", color="indicator", facet_col="indicator", markers=True,
              color_discrete_map=IND_COLORS, labels={"median": "", "year": ""})
fig.update_yaxes(matches=None, showticklabels=True, title=None)
fig.update_xaxes(dtick=2, title=None)
fig.update_traces(line_width=3.2, marker_size=9)
fig.add_vrect(x0=2019.5, x1=2021.5, fillcolor=SHADE, line_width=0, layer="below", row="all", col="all")
polish(fig, "The typical economy, 2018-2024 (median of the seven countries)", "Shaded band = 2020-2021", height=430, legend=False, facets=True)
FIGS["chart_01_typical_economy"] = fig; fig.show()

**What it reveals.** Median GDP growth fell from 2.6% in 2019 to -3.3% in 2020 and rebounded to 6.9% in 2021. Median unemployment rose from 6.5% to 8.0% in 2020 and was back to 6.4% in 2021. Median inflation barely moved in 2020 (2.4%) and rose only in 2022 (6.9%), which indicates two separate shocks two years apart. The median is used so that a single extreme country does not distort the picture.

---
# Theme 1. Economic growth
*How did GDP and economic output change?*

### Chart 2. Six of the seven economies contracted in 2020

In [ ]:
piv = g.reindex(COUNTRIES[::-1]).copy()
piv.columns = piv.columns.astype(str)
fig = px.imshow(piv, text_auto=".1f", aspect="auto", color_continuous_scale="RdBu", color_continuous_midpoint=0, labels=dict(color="GDP growth<br>(%)"))
fig.update_xaxes(title=None); fig.update_yaxes(title=None)
polish(fig, "Annual GDP growth by country (%)", "Red = contraction, blue = growth", height=450, legend=False)
FIGS["chart_02_gdp_growth_heatmap"] = fig; fig.show()

**What it reveals.** In 2020 France (-7.4%), India (-5.8%), Germany (-4.1%), Brazil (-3.3%), Rwanda (-2.8%) and the United States (-2.1%) contracted. China grew by 2.3%, far below its 6.1% in 2019. All seven rebounded in 2021. Germany contracted again in 2023 and 2024 (-0.9% and -0.5%), while India stayed above 7%.

### Chart 3. GDP growth in 2020 compared with each country's own normal

In [ ]:
ge20 = full[full.year == 2020].set_index("country")["gdp_effect"].sort_values().rename("effect").reset_index()
ge20["direction"] = np.where(ge20["effect"] < 0, "Below own normal", "Above own normal")
fig = px.bar(ge20, x="effect", y="country", orientation="h", color="direction",
             color_discrete_map={"Below own normal": BAD, "Above own normal": GOOD}, labels={"effect": "Percentage points", "country": ""})
fig.update_traces(texttemplate="%{x:+.1f}", textposition="outside", cliponaxis=False)
fig.add_vline(x=0, line_color=INK)
fig.update_yaxes(categoryorder="array", categoryarray=list(ge20["country"][::-1]))
polish(fig, "GDP growth in 2020 compared with each country's own 2018-2019 average",
       "Percentage points; negative means growth fell below that country's normal", height=440, legend=False)
FIGS["chart_03_gdp_loss_vs_normal"] = fig; fig.show()

In [ ]:
n_below = int((ge20["effect"] < 0).sum())
ORD = {1: "first", 2: "second", 3: "third", 4: "fourth", 5: "fifth", 6: "sixth", 7: "seventh"}
top_c = ge20["country"].iloc[0]
raw_rank = int(g[2020].rank().loc[top_c])
display(Markdown(f"**What it reveals.** {n_below} of 7 countries grew less in 2020 than their own 2018-2019 average. The largest shortfalls were in "
                 f"{join_names(ge20['country'][:3])} ({', '.join(f'{v:.1f}' for v in ge20['effect'][:3])} points) and the smallest in {nm(ge20['country'].iloc[-1])} "
                 f"({ge20['effect'].iloc[-1]:.1f}). Comparing each country with its own normal matters: {nm(top_c)}'s raw 2020 growth ({g.loc[top_c, 2020]:.1f}%) "
                 f"is only the {ORD[raw_rank]}-lowest of 7, yet its gap to its own normal is the largest."))

---
# Theme 2. Employment
*How did unemployment and jobs change?*

### Chart 4. Unemployment rose most in the United States in 2020

In [ ]:
chg = (u[2020] - u[2019]).rename("change").reset_index().sort_values("change")
chg["direction"] = np.where(chg["change"] >= 0, "Increase", "Decrease")
fig = px.bar(chg, x="change", y="country", orientation="h", color="direction",
             color_discrete_map={"Increase": BAD, "Decrease": GOOD}, labels={"change": "Percentage points", "country": ""})
fig.update_traces(texttemplate="%{x:+.1f}", textposition="outside", cliponaxis=False)
fig.add_vline(x=0, line_color=INK)
fig.update_yaxes(categoryorder="array", categoryarray=list(chg["country"]))
polish(fig, "Change in the unemployment rate between 2019 and 2020", "Percentage points; the United States rose from 3.7% to 8.1%", height=440, legend=False)
FIGS["chart_04_unemployment_change"] = fig; fig.show()

**What it reveals.** The United States recorded by far the largest increase (4.4 points, from 3.7% to 8.1%), followed by Brazil (1.8) and India (1.3). Germany, Rwanda and China changed by less than one point, and France recorded a slight decline of 0.4.

### Chart 5. Unemployment returned to its earlier level at different speeds

In [ ]:
fig = px.line(eco, x="year", y="unemployment_pct", color="country", facet_col="country", facet_col_wrap=4, markers=True,
              color_discrete_map=COUNTRY_COLORS, category_orders={"country": COUNTRIES}, labels={"unemployment_pct": "", "year": ""})
fig.update_yaxes(matches=None, showticklabels=True, title=None)
fig.update_xaxes(dtick=2, title=None)
fig.update_traces(line_width=2.8, marker_size=7)
fig.add_vrect(x0=2019.5, x1=2021.5, fillcolor=SHADE, line_width=0, layer="below", row="all", col="all")
polish(fig, "Unemployment rate by country, 2018-2024 (%)", "Each panel has its own vertical scale; shaded band = 2020-2021", height=560, legend=False, facets=True)
FIGS["chart_05_unemployment_trend"] = fig; fig.show()

In [ ]:
peak_year = u.loc[:, 2020:].idxmax(axis=1)
late_peak = [c for c in COUNTRIES if peak_year[c] > 2020]
returned = [c for c in COUNTRIES if (u.loc[c, 2021:] <= u.loc[c, 2019]).any()]
not_returned = [c for c in COUNTRIES if c not in returned]
at_2024 = [c for c in COUNTRIES if u.loc[c, 2024] <= u.loc[c, 2019]]
late_txt = "; ".join(f"in {nm(c)} the peak came later, in {peak_year[c]} ({u.loc[c, peak_year[c]]:.1f}%)" for c in late_peak)
display(Markdown("**What it reveals.** In most countries unemployment peaked in 2020 and fell afterwards" + (f"; {late_txt}." if late_peak else ".")
                 + f" Unemployment returned to its 2019 level at some point by 2024 in {len(returned)} of 7 countries"
                 + (f" (all except {join_names(not_returned)})" if not_returned else "")
                 + f", although in 2024 itself it was at or below its 2019 level in only {len(at_2024)} ({join_names(at_2024)})."
                 + f" The United States shows the fastest round trip: {u.loc['United States', 2019]:.1f}% in 2019, {u.loc['United States', 2020]:.1f}% in 2020 and {u.loc['United States', 2022]:.1f}% in 2022."))

---
# Theme 3. Prices and inflation
*How did inflation and consumer prices evolve?*

### Chart 6. Inflation rose in 2021-2022 and fell back by 2024

In [ ]:
fig = px.line(eco, x="year", y="inflation_pct", color="country", facet_col="country", facet_col_wrap=4, markers=True,
              color_discrete_map=COUNTRY_COLORS, category_orders={"country": COUNTRIES}, labels={"inflation_pct": "", "year": ""})
fig.update_yaxes(matches=None, showticklabels=True, title=None)
fig.update_xaxes(dtick=2, title=None)
fig.update_traces(line_width=2.8, marker_size=7)
fig.add_vrect(x0=2020.5, x1=2022.5, fillcolor="rgba(230,159,0,0.13)", line_width=0, layer="below", row="all", col="all")
polish(fig, "Annual inflation by country (%)", "Each panel has its own vertical scale; shaded band = 2021-2022", height=560, legend=False, facets=True)
FIGS["chart_06_inflation_trend"] = fig; fig.show()

**What it reveals.** Inflation was low in 2020 (0.1% in Germany and 0.5% in France) and rose markedly in 2021-2022. In 2022 it reached 9.3% in Brazil, 8.0% in the United States and 6.9% in Germany, while Rwanda recorded 17.7% in 2022 and 19.8% in 2023, by far the highest values. China remained below 3% throughout. By 2024 inflation had fallen back to between 0.2% (China) and 5.0% (India). The chart shows when prices rose, not what caused the increase.

### Chart 7. Inflation in 2022 compared with each country's own normal

In [ ]:
ie22 = full[full.year == 2022].set_index("country")["infl_effect"].sort_values().rename("effect").reset_index()
ie22["direction"] = np.where(ie22["effect"] >= 0, "Above own normal", "Below own normal")
fig = px.bar(ie22, x="effect", y="country", orientation="h", color="direction",
             color_discrete_map={"Above own normal": BAD, "Below own normal": GOOD}, labels={"effect": "Percentage points", "country": ""})
fig.update_traces(texttemplate="%{x:+.1f}", textposition="outside", cliponaxis=False)
fig.add_vline(x=0, line_color=INK)
fig.update_yaxes(categoryorder="array", categoryarray=list(ie22["country"]))
polish(fig, "Inflation in 2022 compared with each country's own 2018-2019 average",
       "Percentage points; positive means prices rose faster than that country's normal", height=440, legend=False)
FIGS["chart_07_inflation_vs_normal"] = fig; fig.show()

In [ ]:
top = ie22.sort_values("effect", ascending=False)
n_above = int((ie22["effect"] > 0).sum())
display(Markdown(f"**What it reveals.** In 2022, {n_above} of 7 countries had inflation above their own 2018-2019 average. The largest increases were in "
                 f"{join_names(top['country'][:3])} ({', '.join(f'{v:+.1f}' for v in top['effect'][:3])} points), and the smallest in {nm(top['country'].iloc[-1])} "
                 f"({top['effect'].iloc[-1]:+.1f}). Inflation in 2022 was therefore a widespread but very unequal increase."))

---
# Theme 4. Global differences
*How did the impact vary across countries?*

This theme compares the seven countries on the economic measures, then adds the **health and policy context** (deaths and government restrictions) to see whether the differences in economic impact follow the differences in health toll or strictness.

### Chart 8. Impact scorecard: how each country ranks on four measures

In [ ]:
RANK_COLS = {"GDP growth vs own 2018-19 average, 2020 (pts)": ("GDP loss (2020)", True),
             "Unemployment change, 2019 to 2020 (pts)": ("Unemployment rise (2020)", False),
             "Inflation vs own 2018-19 average, 2022 (pts)": ("Inflation rise (2022)", False),
             "Economy vs 2018-19 trend, 2024 (%)": ("Shortfall vs trend (2024)", True)}
ranks = pd.DataFrame({short: impact[col].rank(ascending=asc).astype(int) for col, (short, asc) in RANK_COLS.items()})
ranks["Average rank"] = ranks.mean(axis=1)
ranks = ranks.sort_values("Average rank")
heat = ranks.drop(columns="Average rank")
fig = px.imshow(heat, text_auto="d", aspect="auto", color_continuous_scale="RdBu", zmin=1, zmax=7, labels=dict(color="Rank"))
fig.update_xaxes(side="top", title=None); fig.update_yaxes(title=None)
polish(fig, "Impact scorecard: rank of each country on four measures",
       "Rank 1 = hit hardest (red), rank 7 = least affected (blue); countries sorted by average rank", height=470, legend=False)
fig.update_layout(margin=dict(t=150))
FIGS["chart_08_impact_scorecard"] = fig; fig.show()
display(style_table(ranks, formats={**{c: "{:d}" for c in heat.columns}, "Average rank": "{:.1f}"}, gradients=[(["Average rank"], "RdBu")]))

In [ ]:
from collections import Counter
worst, best = ranks.index[0], ranks.index[-1]
first_on = {short: heat[short].idxmin() for short in heat.columns}          # country ranked 1 on each measure
LEAD_C, LEAD_N = Counter(first_on.values()).most_common(1)[0]
LEAD_MEASURES = [s_ for s_, c_ in first_on.items() if c_ == LEAD_C]
others = [f"{nm(c_)} ({s_})" for s_, c_ in first_on.items() if c_ != LEAD_C]
display(Markdown(f"**What it reveals.** {Nm(LEAD_C)} is the most affected country on {LEAD_N} of the 4 measures ({', '.join(LEAD_MEASURES)})"
                 + (f"; the other leaders are {', '.join(others)}" if others else "")
                 + f". On average across the four measures, {nm(worst)} ranks as most affected ({ranks['Average rank'].iloc[0]:.1f}) and {nm(best)} as least affected "
                 f"({ranks['Average rank'].iloc[-1]:.1f}). The average rank is descriptive only, since it gives equal weight to four different measures."))

#### Health and policy context

### Chart 9. Reported deaths were concentrated in Brazil and the United States

In [ ]:
fig = px.bar(y, x="country", y="new_deaths_per_million", color="year_str", barmode="group", color_discrete_map=YEAR_COLORS,
             category_orders={"country": COUNTRIES}, labels={"new_deaths_per_million": "Reported deaths per million", "year_str": "Year", "country": ""})
fig.update_traces(texttemplate="%{y:,.0f}", textposition="outside", cliponaxis=False, textfont_size=10)
polish(fig, "Reported COVID-19 deaths per million, by country and year",
       "Reported figures depend on testing and registration rules, so they are not fully comparable across countries", height=500)
FIGS["chart_09_deaths_by_year"] = fig; fig.show()

**What it reveals.** Brazil in 2021 recorded the highest figure (2,034 per million), followed by the United States in 2021 (1,375). France was highest in 2020 (975) and Germany in 2021 (858). India (104, 233 and 35 per million), Rwanda (6, 93 and 9) and China (3, 1 and 27) report far lower figures, which partly reflects differences in testing and reporting and should not be read as a milder epidemic.

### Chart 10. Deaths peaked at different times in each country

In [ ]:
fig = px.area(m, x="date", y="new_deaths_per_million", color="country", facet_col="country", facet_col_wrap=4,
              color_discrete_map=COUNTRY_COLORS, category_orders={"country": COUNTRIES}, labels={"new_deaths_per_million": "Deaths per million", "date": ""})
fig.update_yaxes(matches=None, showticklabels=True, title=None)
fig.update_xaxes(title=None, dtick="M12", tickformat="%Y")
fig.update_traces(line_width=1.8)
polish(fig, "Reported COVID-19 deaths per million, month by month",
       "Each panel has its own vertical scale: compare the timing of the waves, not the heights", height=600, legend=False, facets=True)
FIGS["chart_10_deaths_waves"] = fig; fig.show()

**What it reveals.** The waves were not synchronized. France (November 2020, 317 per million) and Germany (December 2020, 278) peaked first, the United States in January 2021 (284), Brazil in April 2021 (400, the highest of all), India in May 2021 (84) and Rwanda in July 2021 (26). China's highest month was December 2022 (10), after its restrictions were lifted.

### Chart 11. Restrictions peaked in spring 2020 and eased in 2022, except in China

In [ ]:
hm = m.pivot(index="country", columns="month", values="sti_weighted").reindex(COUNTRIES[::-1])
fig = px.imshow(hm, aspect="auto", color_continuous_scale="YlOrRd", zmin=0, zmax=100, labels=dict(color="Stringency<br>(0-100)"))
fig.update_xaxes(tickmode="array", tickvals=list(hm.columns[::3]), tickangle=-45, title=None)
fig.update_yaxes(title=None)
polish(fig, "Stringency of government restrictions, month by month", "Index from 0 (no restrictions) to 100 (strictest); darker means stricter", height=470, legend=False)
FIGS["chart_11_stringency_heatmap"] = fig; fig.show()

**What it reveals.** Most countries reached their strictest month in spring 2020: India 98.6, Rwanda 90.7, France 88.0, Brazil 80.5, China 79.5 and the United States 72.7. Germany peaked later, at 83.5 in January 2021. By December 2022 France and Germany had fallen to 11.1, while China was still at 60.6, and China's yearly average rose from 68.3 in 2020 to 77.6 in 2022, the only country where restrictions were stricter in 2022 than in 2020.

### Chart 12. Rules for non-vaccinated people became stricter than for vaccinated people

In [ ]:
fig = px.line(m, x="date", y="sti_gap", color="country", color_discrete_map=COUNTRY_COLORS, category_orders={"country": COUNTRIES},
              labels={"sti_gap": "Index points", "date": ""})
fig.update_traces(line_width=2.6)
fig.update_xaxes(dtick="M6", tickformat="%b %Y")
polish(fig, "Gap in restriction strictness: non-vaccinated minus vaccinated people", "Zero means identical rules for both groups", height=500)
FIGS["chart_12_policy_gap"] = fig; fig.show()

**What it reveals.** The gap is zero everywhere in 2020, before vaccines existed. It opened at different dates in 2021: the United States in April, France in June, Germany and India in August, Brazil in September and Rwanda in December. France reached the largest gap in 2021 (45.4 points, September) and Rwanda in 2022 (44.5, October). China shows no gap at any point.

#### Do the differences in economic impact follow the health and policy differences?

### Chart 13. In 2020 the economic loss did not follow the reported death toll

In [ ]:
c20 = cov[cov.year == 2020].copy()
POS = {"Brazil": "top center", "United States": "bottom center", "Germany": "middle left", "France": "top center", "India": "top center", "Rwanda": "top center", "China": "top center"}
fig = px.scatter(c20, x="new_deaths_per_million", y="gdp_effect", color="country", text="country", log_x=True, color_discrete_map=COUNTRY_COLORS,
                 labels={"new_deaths_per_million": "Reported deaths per million, 2020 (logarithmic scale)", "gdp_effect": "GDP growth minus 2018-19 average (points)"})
fig.update_traces(marker=dict(size=16, line=dict(width=1.5, color="white")), textfont_size=12)
fig.for_each_trace(lambda t: t.update(textposition=POS.get(t.name, "top center")))
fig.add_hline(y=0, line_color=INK)
polish(fig, "Economic loss versus reported deaths, 2020", "Each dot is one country; lower on the chart means a larger fall below its own normal growth", height=520, legend=False)
FIGS["chart_13_loss_vs_deaths_2020"] = fig; fig.show()

**What it reveals.** The largest economic losses in 2020 were in Rwanda (-11.5 points) and India (-10.9), which reported very few deaths (6 and 104 per million). France (-9.3), Germany (-5.2), the United States (-4.9) and Brazil (-4.8) reported 540 to 1,009 deaths per million and lost less relative to their own normal. China reported almost no deaths and lost 4.1 points. There is no visible relationship between the two measures (Spearman +0.07, 7 countries).

### Chart 14. Restriction strictness and economic impact, 2020-2022

In [ ]:
fig = px.scatter(cov, x="sti_weighted", y="gdp_effect", color="country", text="year_str", color_discrete_map=COUNTRY_COLORS,
                 labels={"sti_weighted": "Stringency index (yearly average, 0-100)", "gdp_effect": "GDP growth minus 2018-19 average (points)"})
fig.update_traces(marker=dict(size=13, line=dict(width=1, color="white")), textposition="top center", textfont_size=10)
sub = cov[["sti_weighted", "gdp_effect"]].dropna()
slope, intercept = np.polyfit(sub.sti_weighted, sub.gdp_effect, 1)
xs = np.linspace(sub.sti_weighted.min(), sub.sti_weighted.max(), 50)
fig.add_trace(go.Scatter(x=xs, y=intercept + slope * xs, mode="lines", name="Fitted line (all points)", line=dict(color=INK, dash="dot", width=2)))
fig.add_hline(y=0, line_color="#9CA3AF")
polish(fig, "Restriction strictness versus economic impact, 2020-2022", "21 points = 7 countries x 3 years; the label on each dot is the year", height=540)
FIGS["chart_14_strictness_vs_loss"] = fig; fig.show()

**What it reveals.** No clear pattern is visible (Spearman -0.10, p = 0.68). The points separate by year more than by strictness: 2020 lies below zero for every country, 2021 above zero and 2022 close to zero. The fitted line summarizes the overall tendency and does not indicate cause.

#### Statistical tests

Spearman rank correlation compares rankings, so extreme values (for example China's near-zero case counts) do not distort it. The coefficient runs from -1 to +1, and a p-value below 0.05 indicates a pattern unlikely to be due to chance. The sample is small (21 country-years from 7 countries, each appearing three times), so the p-values are optimistic and the results are exploratory.

In [ ]:
def spear(df, x, yv):
    s = df[[x, yv]].dropna()
    rho, p = spearmanr(s[x], s[yv])
    return rho, p, len(s)

NAMES = {"sti_weighted": "Restriction strictness", "new_deaths_per_million": "Reported deaths", "new_cases_per_million": "Reported cases",
         "gdp_effect": "GDP effect", "unemp_effect": "Unemployment effect", "infl_effect": "Inflation effect"}
pairs = [("sti_weighted", "gdp_effect"), ("new_deaths_per_million", "gdp_effect"), ("new_cases_per_million", "gdp_effect"),
         ("sti_weighted", "unemp_effect"), ("new_deaths_per_million", "unemp_effect"), ("sti_weighted", "infl_effect")]
rows = []
for a, b in pairs:
    rho, p, n = spear(cov, a, b)
    strength = "very weak" if abs(rho) < 0.1 else "weak" if abs(rho) < 0.3 else "moderate" if abs(rho) < 0.5 else "strong"
    rows.append({"Relationship": f"{NAMES[a]} vs {NAMES[b]}", "rho": rho, "p-value": p, "n": n,
                 "Reading": f"{strength} {'positive' if rho > 0 else 'negative'}, " + ("significant at 5%" if p < 0.05 else "not significant")})
results = pd.DataFrame(rows).set_index("Relationship")
display(results.style.format({"rho": "{:+.2f}", "p-value": "{:.3f}", "n": "{:d}"})
        .map(lambda v: "background-color:#FDE68A;font-weight:600" if isinstance(v, float) and v < 0.05 else "", subset=["p-value"])
        .set_table_styles([{"selector": "th", "props": [("background-color", "#F3F4F6"), ("color", INK), ("font-weight", "600")]}]))

rob = []
for yr in (2020, 2021, 2022):
    s = cov[cov.year == yr]
    for a in ("sti_weighted", "new_deaths_per_million"):
        rho, p, n = spear(s, a, "gdp_effect")
        rob.append({"Year": yr, "Relationship": f"{NAMES[a]} vs GDP effect", "rho": rho, "p-value": p, "n": n})
rob = pd.DataFrame(rob).set_index(["Year", "Relationship"])
print("Robustness check: the same two relationships, year by year (7 countries each, a rough guide only)")
display(rob.style.format({"rho": "{:+.2f}", "p-value": "{:.3f}", "n": "{:d}"}))

---
# Theme 5. Economic recovery
*How did economies recover after the initial shock?*

### Chart 15. Has each economy regained its 2019 size?

In [ ]:
fig = px.line(idx, x="year", y="gdp_index", color="country", markers=True, color_discrete_map=COUNTRY_COLORS, category_orders={"country": COUNTRIES},
              labels={"gdp_index": "Size of the economy (2019 = 100)", "year": ""})
fig.update_traces(line_width=2.8, marker_size=8)
fig.add_hline(y=100, line_dash="dash", line_color=INK, annotation_text="2019 level", annotation_position="bottom right")
fig.update_xaxes(dtick=1)
polish(fig, "Size of each economy relative to its own 2019 level", "Above the dashed line = larger than in 2019; each country is compared only with itself", height=500)
FIGS["chart_15_economy_index"] = fig; fig.show()

**What it reveals.** Six of the seven economies fell below their 2019 size in 2020, most sharply France (92.6) and India (94.2); China (102.3) did not. France and Germany were still below 2019 in 2021 (98.9 and 99.6), because a rebound applies to a smaller economy. By 2022 all seven were above their 2019 size. By 2024 Rwanda (134.7), India (127.7) and China (126.8) were more than a quarter larger than in 2019, while Germany had only just returned to 100.

### Chart 16. Actual economy versus its 2018-2019 growth trend

In [ ]:
long = idx.melt(id_vars=["country", "year"], value_vars=["gdp_index", "trend_index"], var_name="path", value_name="index")
long["path"] = long["path"].map({"gdp_index": "Actual", "trend_index": "2018-19 growth trend (estimate)"})
fig = px.line(long, x="year", y="index", color="path", line_dash="path", facet_col="country", facet_col_wrap=4, category_orders={"country": COUNTRIES},
              color_discrete_map={"Actual": "#0072B2", "2018-19 growth trend (estimate)": "#9CA3AF"},
              line_dash_map={"Actual": "solid", "2018-19 growth trend (estimate)": "dash"}, labels={"index": "", "year": ""})
fig.update_traces(line_width=2.8)
fig.update_xaxes(dtick=2, title=None); fig.update_yaxes(title=None)
polish(fig, "Actual economy versus the path implied by 2018-2019 growth",
       "Size of the economy, 2019 = 100; the gap between the lines is an estimate of the cost of the pandemic", height=560, facets=True)
FIGS["chart_16_actual_vs_trend"] = fig; fig.show()

**What it reveals.** By 2024, relative to the 2018-2019 trend, Rwanda's economy was approximately 11% below its projected level, China's 7%, Germany's and France's 5% each, the United States' 2% and India's 1%, while Brazil's exceeded its projected level by approximately 3%. The projected path is an estimate based on two years of pre-pandemic data.

**Table 5. Recovery by country.**

In [ ]:
rec = []
for c in COUNTRIES:
    gi = idx[idx.country == c].set_index("year")["gdp_index"]
    below = gi[(gi.index >= 2020) & (gi < 100)]
    if below.empty:
        gdp_back = "never below"
    else:
        later = gi[(gi.index > below.index.min()) & (gi >= 100)]
        gdp_back = str(int(later.index.min())) if len(later) else "not yet"
    u_after = u.loc[c, 2021:]
    u_ok = u_after[u_after <= u.loc[c, 2019]]
    rec.append({"Country": c,
                "GDP back above 2019 size": gdp_back,
                "Unemployment back to 2019 level": str(int(u_ok.index.min())) if len(u_ok) else "not yet",
                "Economy vs 2018-19 trend, 2024 (%)": idx[(idx.country == c) & (idx.year == 2024)]["gap_pct"].iloc[0],
                "Inflation 2024 (%)": inf.loc[c, 2024]})
t5 = pd.DataFrame(rec).set_index("Country")
display(style_table(t5, formats={"Economy vs 2018-19 trend, 2024 (%)": "{:+.1f}", "Inflation 2024 (%)": "{:.1f}"},
                    gradients=[(["Economy vs 2018-19 trend, 2024 (%)"], "RdBu"), (["Inflation 2024 (%)"], "Oranges")]))

In [ ]:
no_dip = [c for c in COUNTRIES if t5.loc[c, "GDP back above 2019 size"] == "never below"]
late_gdp = [c for c in COUNTRIES if t5.loc[c, "GDP back above 2019 size"] == "2022"]
u_back = [c for c in COUNTRIES if t5.loc[c, "Unemployment back to 2019 level"] != "not yet"]
u_not = [c for c in COUNTRIES if t5.loc[c, "Unemployment back to 2019 level"] == "not yet"]
display(Markdown(f"**What it reveals.** GDP recovery was fast in most countries: {join_names([c for c in COUNTRIES if t5.loc[c, 'GDP back above 2019 size'] == '2021'])} were back above their 2019 size in 2021, "
                 f"and {join_names(late_gdp)} only in 2022 ({join_names(no_dip)} never fell below). Unemployment returned to its 2019 level in {len(u_back)} of 7 countries at some point by 2024"
                 + (f"; it had not returned to its 2019 level in {join_names(u_not)}." if u_not else ".")
                 + f" Against the estimated pre-COVID trend, {int((t5['Economy vs 2018-19 trend, 2024 (%)'] < 0).sum())} of 7 economies were still below in 2024, so recovery of the 2019 size did not mean recovery of the lost growth."))

---
# Findings for the five themes

The text below is **generated from the data** by the code in the next cells, so it always matches the tables and charts above. Every statement names the countries it refers to.

In [ ]:
countries_txt = ", ".join(COUNTRIES)

# ---------- Theme 1: Economic growth ----------
med_g = g.median(); n_neg = int((g[2020] < 0).sum())
pos20 = [c for c in COUNTRIES if g.loc[c, 2020] >= 0]
ge = full[full.year == 2020].set_index("country")["gdp_effect"].sort_values()
f1 = [
 f"Median GDP growth across the seven countries fell from {med_g[2019]:.1f}% in 2019 to {med_g[2020]:.1f}% in 2020 and rebounded to {med_g[2021]:.1f}% in 2021.",
 f"{n_neg} of 7 economies contracted in 2020 ({join_names([c for c in COUNTRIES if g.loc[c, 2020] < 0])}); {join_names(pos20)} stayed positive ({g.loc[pos20[0], 2020]:+.1f}%) but far below its 2019 rate ({g.loc[pos20[0], 2019]:.1f}%).",
 f"Against their own 2018-2019 average, the largest shortfalls in 2020 were in {join_names(ge.index[:3])} ({', '.join(f'{v:.1f}' for v in ge.iloc[:3])} points); all 7 countries grew less than their own normal."]

# ---------- Theme 2: Employment ----------
uc = (u[2020] - u[2019]).sort_values(ascending=False)
peak_year = u.loc[:, 2020:].idxmax(axis=1); late_peak = [c for c in COUNTRIES if peak_year[c] > 2020]
u_med = u.median()
late_sentence = (("In " + "; in ".join(f"{nm(c)} the peak came later, in {peak_year[c]} ({u.loc[c, peak_year[c]]:.1f}%)" for c in late_peak) + ".")
                 if late_peak else "Unemployment peaked in 2020 in every country.")
f2 = [
 f"Unemployment rose most between 2019 and 2020 in {nm(uc.index[0])} ({uc.iloc[0]:+.1f} points, from {u.loc[uc.index[0], 2019]:.1f}% to {u.loc[uc.index[0], 2020]:.1f}%), followed by {nm(uc.index[1])} ({uc.iloc[1]:+.1f}) and {nm(uc.index[2])} ({uc.iloc[2]:+.1f}); it fell in {join_names(uc[uc < 0].index)}.",
 f"Median unemployment across the seven countries went from {u_med[2019]:.1f}% in 2019 to {u_med[2020]:.1f}% in 2020 and {u_med[2021]:.1f}% in 2021.",
 late_sentence]

# ---------- Theme 3: Prices and inflation ----------
imax = inf.stack().idxmax()
ie = full[full.year == 2022].set_index("country")["infl_effect"].sort_values(ascending=False)
f3 = [
 f"Median inflation across the seven countries was {inf.median()[2020]:.1f}% in 2020 and {inf.median()[2022]:.1f}% in 2022: prices rose after the output and jobs shock, not during it.",
 f"The largest rises in 2022 against each country's own 2018-2019 average were in {join_names(ie.index[:3])} ({', '.join(f'{v:+.1f}' for v in ie.iloc[:3])} points); the highest inflation recorded was {inf.loc[imax]:.1f}% ({nm(imax[0])}, {imax[1]}).",
 f"{join_names([c for c in COUNTRIES if inf.loc[c].max() < 3])} stayed below 3% throughout 2018-2024."]

# ---------- Theme 4: Global differences ----------
dt = y.groupby("country")["new_deaths_per_million"].sum().sort_values(ascending=False)
pk = m.loc[m.groupby("country")["new_deaths_per_million"].idxmax()].sort_values("date")
ym = y.pivot(index="country", columns="year", values="sti_weighted")
rose = [c for c in ym.index if ym.loc[c, 2022] > ym.loc[c, 2020]]
sig = results[results["p-value"] < 0.05]
r20 = spear(cov[cov.year == 2020], "new_deaths_per_million", "gdp_effect")
d20 = y[y.year == 2020].set_index("country")["new_deaths_per_million"]
f4 = [
 f"{Nm(LEAD_C)} is the most affected country on {LEAD_N} of the 4 economic measures ({', '.join(LEAD_MEASURES)}); on average across the four, {nm(ranks.index[0])} ranks as most affected and {nm(ranks.index[-1])} as least affected (descriptive ranking, Chart 8).",
 f"Reported deaths over 2020-2022 were highest in {nm(dt.index[0])} ({fmt0(dt.iloc[0])} per million) and {nm(dt.index[1])} ({fmt0(dt.iloc[1])}), and lowest in {nm(dt.index[-1])} ({fmt0(dt.iloc[-1])}) and {nm(dt.index[-2])} ({fmt0(dt.iloc[-2])}); the deadliest month ranged from {mon(pk.iloc[0]['date'])} ({nm(pk.iloc[0]['country'])}) to {mon(pk.iloc[-1]['date'])} ({nm(pk.iloc[-1]['country'])}).",
 f"Restrictions eased between 2020 and 2022 in {7 - len(rose)} countries and rose only in {join_names(rose)} (average stringency {ym.loc[rose[0], 2020]:.1f} to {ym.loc[rose[0], 2022]:.1f}).",
 f"The economic loss did not follow the death toll: in 2020, {nm(ge.index[0])} and {nm(ge.index[1])} had the largest losses with {d20[ge.index[0]]:,.0f} and {d20[ge.index[1]]:,.0f} reported deaths per million, against {d20['France']:,.0f} in France and {d20['United States']:,.0f} in the United States (rank correlation {r20[0]:+.2f}, n = {r20[2]}).",
 f"Across 2020-2022, {len(sig)} of {len(results)} correlation tests between health or policy and economic outcomes reached p < 0.05" + (f" ({', '.join(sig.index)}, most likely a timing effect since restrictions eased in 2022, when inflation peaked)." if len(sig) else ".")]

# ---------- Theme 5: Economic recovery ----------
i21 = idx[idx.year == 2021].set_index("country")["gdp_index"]
below21 = list(i21[i21 < 100].index)
gap24 = idx[idx.year == 2024].set_index("country")["gap_pct"].sort_values()
low_inf = [c for c in COUNTRIES if inf.loc[c, 2024] < 3]
above = [c for c in COUNTRIES if gap24[c] > 0]
f5 = [
 f"All 7 economies were above their 2019 size by 2022; {join_names(below21)} were still below it in 2021, and {join_names(no_dip)} never fell below it.",
 f"By 2024, {int((gap24 < 0).sum())} of 7 economies remained below the estimated 2018-2019 growth trend, the largest shortfall being {nm(gap24.index[0])} ({gap24.iloc[0]:.0f}%); {join_names(above)} {'was' if len(above) == 1 else 'were'} above it ({gap24.iloc[-1]:+.0f}%).",
 f"Unemployment returned to its 2019 level at some point by 2024 in {len(u_back)} of 7 countries" + (f" (all except {join_names(u_not)})" if u_not else "") + f", and inflation was below 3% in 2024 in {join_names(low_inf)}."]

THEMES = [("1. Economic growth", f1), ("2. Employment", f2), ("3. Prices and inflation", f3), ("4. Global differences", f4), ("5. Economic recovery", f5)]
for title, items in THEMES:
    display(Markdown(f"### Theme {title}\n" + "\n".join(f"- {t}" for t in items)))

## Limits of the analysis (for the critical-analysis section)

- **Seven selected countries, not the world.** The dashboard question concerns the global economy, but the data covers Brazil, China, France, Germany, India, Rwanda and the United States only; results describe these countries.
- **Short health and policy window.** Health and policy data cover 2020-2022, so links with the economy rest on 21 country-years.
- **Short baseline.** The pre-COVID reference is two years (2018-2019), so baselines and the trend path are estimates.
- **Reporting differences.** Case and death counts depend on testing and registration rules and are not fully comparable across countries.
- **Association, not cause.** Correlations do not show that restrictions or deaths caused economic outcomes; the 2021 rebound is partly a base effect, and the 2022 price rise overlaps with other events.

## Short findings document and exports

The next cell writes `findings_summary.md`: the countries and scope, a headline table with one row per theme, the findings for each theme, a country summary table and the limits. It also saves every table (CSV) and every chart (specification as JSON, plus an interactive HTML file) in an `outputs` folder.

In [ ]:
os.makedirs("outputs/charts", exist_ok=True)

# ---- headline table: one row per theme ----
late_txt2 = ", ".join(f"{nm(c)} ({u.loc[c, peak_year[c]]:.1f}% in {peak_year[c]})" for c in late_peak)
low_inf_all = [c for c in COUNTRIES if inf.loc[c].max() < 3]
headlines = pd.DataFrame([
 [f"Median GDP growth fell from {med_g[2019]:.1f}% (2019) to {med_g[2020]:.1f}% (2020), then rebounded to {med_g[2021]:.1f}% (2021); {n_neg} of 7 economies contracted in 2020.",
  f"Largest shortfalls vs own normal: {join_names(ge.index[:3])} ({', '.join(f'{v:.1f}' for v in ge.iloc[:3])} pts)", f"All 7; still growing in 2020: {join_names(pos20)}"],
 [f"Unemployment rose most in {nm(uc.index[0])} ({uc.iloc[0]:+.1f} pts) and fell in {join_names(uc[uc < 0].index)}; the median went {u_med[2019]:.1f}% to {u_med[2020]:.1f}% to {u_med[2021]:.1f}%.",
  (f"Later peaks: {late_txt2}" if late_peak else "Peak in 2020 in every country"), "All 7"],
 [f"Inflation was low in 2020 ({inf.median()[2020]:.1f}% median) and peaked in 2022 ({inf.median()[2022]:.1f}% median).",
  f"Highest: {inf.loc[imax]:.1f}% ({nm(imax[0])}, {imax[1]}); largest 2022 rises: {join_names(ie.index[:3])}", f"All 7; below 3% throughout: {join_names(low_inf_all)}"],
 [f"Impact differed widely: {nm(ranks.index[0])} is most affected on average and {nm(ranks.index[-1])} least; the economic loss did not follow reported deaths or restriction strictness.",
  f"Deaths vs GDP loss in 2020: rho {r20[0]:+.2f} (n = {r20[2]}); {nm(LEAD_C)} most affected on {LEAD_N} of 4 measures", f"All 7; highest reported deaths: {join_names(dt.index[:2])}; restrictions rose only in {join_names(rose)}"],
 [f"All 7 economies were above their 2019 size by 2022, but {int((gap24 < 0).sum())} of 7 remained below their pre-COVID growth trend in 2024.",
  f"Largest shortfall: {nm(gap24.index[0])} ({gap24.iloc[0]:.0f}%); above trend: {join_names(above)} ({gap24.iloc[-1]:+.0f}%)", "All 7"],
], index=["1. Economic growth", "2. Employment", "3. Prices and inflation", "4. Global differences", "5. Economic recovery"],
   columns=["Main finding", "Key figures", "Countries"])

# ---- country summary table ----
summary = pd.DataFrame({
    "GDP vs own normal, 2020 (pts)": impact.iloc[:, 0].map("{:+.1f}".format),
    "Unemployment change 2019-20 (pts)": impact.iloc[:, 1].map("{:+.1f}".format),
    "Inflation vs own normal, 2022 (pts)": impact.iloc[:, 2].map("{:+.1f}".format),
    "Economy vs trend, 2024 (%)": impact.iloc[:, 3].map("{:+.1f}".format),
    "Reported deaths per million, 2020-22": t3["Reported deaths per million, 2020-22"].map("{:,.0f}".format),
    "Average stringency (0-100)": t3["Average stringency (0-100)"].map("{:.1f}".format),
    "Average impact rank (1 = hardest hit)": ranks["Average rank"].map("{:.1f}".format)}).reindex(ranks.index)

lines = ["# Findings summary: COVID-19 and the economy", "",
         f"**Countries studied (7):** {countries_txt}.",
         "**Period:** economy 2018-2024; health and policy 2020-2022.",
         "**Scope:** the dashboard question concerns the global economy, but these findings describe the seven countries above only.", "",
         "## Headline findings by theme", "", md_table(headlines, "Theme"), ""]
for title, items in THEMES:
    lines += [f"## Theme {title}"] + [f"- {t}" for t in items] + [""]
lines += ["## Country summary table", "",
          "*Each economic measure is compared with the country's own 2018-2019 average or trend; countries are sorted from most to least affected on average.*", "",
          md_table(summary, "Country"), "",
          "## Limits",
          "- Seven selected countries, not the world; health and policy data cover 2020-2022 only (21 country-years).",
          "- The pre-COVID baseline and trend are estimates based on 2018-2019.",
          "- Reported cases and deaths are not fully comparable across countries.",
          "- Associations do not establish cause."]
with open("outputs/findings_summary.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

tables = {"table1_typical_economy_by_period": t1, "table2_economy_by_country": t2, "table3_health_policy_by_country": t3,
          "table4_economic_impact": impact, "table5_recovery": t5, "impact_scorecard_ranks": ranks, "spearman_results": results,
          "country_summary": summary, "headline_findings": headlines}
for name, tbl in tables.items():
    tbl.to_csv(f"outputs/{name}.csv")
for name, fg in FIGS.items():
    fg.write_json(f"outputs/charts/{name}.json")
    fg.write_html(f"outputs/charts/{name}.html", include_plotlyjs="cdn")

print("Saved in ./outputs :", sorted(os.listdir("outputs")))
print("Charts             :", len(os.listdir("outputs/charts")), "files")
display(Markdown("\n".join(lines)))